# A small graph RAG with networkx

This notebook runs the library blocks of the lesson *Build a small graph RAG*. No GPU, no API key and no language model are needed: extraction uses rule-based stand-ins. The outputs saved in the cells come from a real run with the pinned version.

In [ ]:
!pip install -q networkx==3.6.1
print('installed')

## 1. Build the graph with networkx 3.6.1

In [ ]:
import re
import networkx as nx

PASSAGES = [
    {"id": "p01", "text": "The Search service is owned by the Atlas team."},
    {"id": "p02", "text": "The Billing service is owned by the Ledger team."},
    {"id": "p03", "text": "The Export service is owned by the Ledger team."},
    {"id": "p04", "text": "The Login service is owned by the Identity team."},
    {"id": "p05", "text": "The Atlas team is led by Priya Nair."},
    {"id": "p06", "text": "The Ledger team is led by Rahul Mehta."},
    {"id": "p07", "text": "The Identity team is led by Sara Khan."},
    {"id": "p08", "text": "Priya Nair is based in Pune."},
    {"id": "p09", "text": "Rahul Mehta is based in Mumbai."},
    {"id": "p10", "text": "Sara Khan is based in Delhi."},
    {"id": "p11", "text": "The Search service depends on the Login service."},
    {"id": "p12", "text": "The Billing service depends on the Login service."},
    {"id": "p13", "text": "The Export service depends on the Billing service."},
    {"id": "p14", "text": "Incident INC-7 affected the Login service."},
    {"id": "p15", "text": "Incident INC-9 affected the Export service."},
    {"id": "p16", "text": "SSO is another name for the Login service."},
    {"id": "p17", "text": "The Export service depends on the Search service."},
]
TEXT = {p["id"]: p["text"] for p in PASSAGES}
ALIASES = {"SSO": "Login", "Search API": "Search"}

NAME = r"([A-Z][\w-]*(?: [A-Z]\w+)?)"      # one capitalised name, or two such as a person
PATTERNS = [                              # stand-in for an extraction model: one rule per relation
    ("owned_by",   rf"The {NAME} service is owned by the {NAME} team"),
    ("led_by",     rf"The {NAME} team is led by {NAME}"),
    ("based_in",   rf"{NAME} is based in {NAME}"),
    ("depends_on", rf"The {NAME} service depends on the {NAME} service"),
    ("affected",   rf"Incident {NAME} affected the {NAME} service"),
]

def extract(passages):
    triples = []
    for p in passages:
        for relation, pattern in PATTERNS:
            for subject, obj in re.findall(pattern, p["text"]):
                triples.append((subject, relation, obj, p["id"]))
    return triples

g = nx.MultiDiGraph()                       # allows two different relations between one pair
for s, relation, o, pid in extract(PASSAGES):
    g.add_edge(s, o, relation=relation, source=pid)

print("nodes", g.number_of_nodes(), "edges", g.number_of_edges())
print("out of Export:", sorted((o, d["relation"]) for _, o, d in g.out_edges("Export", data=True)))
print("successors   :", sorted(g.successors("Export")))
print("predecessors :", sorted(g.predecessors("Login")))

nodes 15 edges 16
out of Export: [('Billing', 'depends_on'), ('Ledger', 'owned_by'), ('Search', 'depends_on')]
successors   : ['Billing', 'Ledger', 'Search']
predecessors : ['Billing', 'INC-7', 'Search']


## 2. Walk it

In [ ]:
import re
import networkx as nx

PASSAGES = [
    {"id": "p01", "text": "The Search service is owned by the Atlas team."},
    {"id": "p02", "text": "The Billing service is owned by the Ledger team."},
    {"id": "p03", "text": "The Export service is owned by the Ledger team."},
    {"id": "p04", "text": "The Login service is owned by the Identity team."},
    {"id": "p05", "text": "The Atlas team is led by Priya Nair."},
    {"id": "p06", "text": "The Ledger team is led by Rahul Mehta."},
    {"id": "p07", "text": "The Identity team is led by Sara Khan."},
    {"id": "p08", "text": "Priya Nair is based in Pune."},
    {"id": "p09", "text": "Rahul Mehta is based in Mumbai."},
    {"id": "p10", "text": "Sara Khan is based in Delhi."},
    {"id": "p11", "text": "The Search service depends on the Login service."},
    {"id": "p12", "text": "The Billing service depends on the Login service."},
    {"id": "p13", "text": "The Export service depends on the Billing service."},
    {"id": "p14", "text": "Incident INC-7 affected the Login service."},
    {"id": "p15", "text": "Incident INC-9 affected the Export service."},
    {"id": "p16", "text": "SSO is another name for the Login service."},
    {"id": "p17", "text": "The Export service depends on the Search service."},
]
TEXT = {p["id"]: p["text"] for p in PASSAGES}
ALIASES = {"SSO": "Login", "Search API": "Search"}

NAME = r"([A-Z][\w-]*(?: [A-Z]\w+)?)"      # one capitalised name, or two such as a person
PATTERNS = [                              # stand-in for an extraction model: one rule per relation
    ("owned_by",   rf"The {NAME} service is owned by the {NAME} team"),
    ("led_by",     rf"The {NAME} team is led by {NAME}"),
    ("based_in",   rf"{NAME} is based in {NAME}"),
    ("depends_on", rf"The {NAME} service depends on the {NAME} service"),
    ("affected",   rf"Incident {NAME} affected the {NAME} service"),
]

def extract(passages):
    triples = []
    for p in passages:
        for relation, pattern in PATTERNS:
            for subject, obj in re.findall(pattern, p["text"]):
                triples.append((subject, relation, obj, p["id"]))
    return triples

g = nx.MultiDiGraph()
for s, relation, o, pid in extract(PASSAGES):
    g.add_edge(s, o, relation=relation, source=pid)

# a view with one relation only: arrows point from a service to what it needs
deps = nx.DiGraph((s, o) for s, o, d in g.edges(data=True) if d["relation"] == "depends_on")
print("needs Login   :", sorted(nx.ancestors(deps, "Login")))
print("Export needs  :", sorted(nx.descendants(deps, "Export")))

path = nx.shortest_path(g.to_undirected(), "INC-9", "Sara Khan")
print("edges in path :", len(path) - 1)
print("ends          :", path[:2], "...", path[-2:])

needs Login   : ['Billing', 'Export', 'Search']
Export needs  : ['Billing', 'Login', 'Search']
edges in path : 5
ends          : ['INC-9', 'Export'] ... ['Identity', 'Sara Khan']


## 3. Groups for questions about the whole collection

In [ ]:
import re
import networkx as nx

PASSAGES = [
    {"id": "p01", "text": "The Search service is owned by the Atlas team."},
    {"id": "p02", "text": "The Billing service is owned by the Ledger team."},
    {"id": "p03", "text": "The Export service is owned by the Ledger team."},
    {"id": "p04", "text": "The Login service is owned by the Identity team."},
    {"id": "p05", "text": "The Atlas team is led by Priya Nair."},
    {"id": "p06", "text": "The Ledger team is led by Rahul Mehta."},
    {"id": "p07", "text": "The Identity team is led by Sara Khan."},
    {"id": "p08", "text": "Priya Nair is based in Pune."},
    {"id": "p09", "text": "Rahul Mehta is based in Mumbai."},
    {"id": "p10", "text": "Sara Khan is based in Delhi."},
    {"id": "p11", "text": "The Search service depends on the Login service."},
    {"id": "p12", "text": "The Billing service depends on the Login service."},
    {"id": "p13", "text": "The Export service depends on the Billing service."},
    {"id": "p14", "text": "Incident INC-7 affected the Login service."},
    {"id": "p15", "text": "Incident INC-9 affected the Export service."},
    {"id": "p16", "text": "SSO is another name for the Login service."},
    {"id": "p17", "text": "The Export service depends on the Search service."},
]
TEXT = {p["id"]: p["text"] for p in PASSAGES}
ALIASES = {"SSO": "Login", "Search API": "Search"}

NAME = r"([A-Z][\w-]*(?: [A-Z]\w+)?)"      # one capitalised name, or two such as a person
PATTERNS = [                              # stand-in for an extraction model: one rule per relation
    ("owned_by",   rf"The {NAME} service is owned by the {NAME} team"),
    ("led_by",     rf"The {NAME} team is led by {NAME}"),
    ("based_in",   rf"{NAME} is based in {NAME}"),
    ("depends_on", rf"The {NAME} service depends on the {NAME} service"),
    ("affected",   rf"Incident {NAME} affected the {NAME} service"),
]

def extract(passages):
    triples = []
    for p in passages:
        for relation, pattern in PATTERNS:
            for subject, obj in re.findall(pattern, p["text"]):
                triples.append((subject, relation, obj, p["id"]))
    return triples

g = nx.MultiDiGraph()
for s, relation, o, pid in extract(PASSAGES):
    g.add_edge(s, o, relation=relation, source=pid)

def groups(graph):
    parts = nx.connected_components(graph.to_undirected())
    return sorted((sorted(part) for part in parts), key=lambda part: part[0])

print("all relations :", [len(p) for p in groups(g)])

kept = g.copy()
kept.remove_edges_from([(s, o, k) for s, o, k, d in g.edges(keys=True, data=True)
                        if d["relation"] == "depends_on"])
for part in groups(kept):
    print(len(part), part)

all relations : [15]
4 ['Atlas', 'Priya Nair', 'Pune', 'Search']
6 ['Billing', 'Export', 'INC-9', 'Ledger', 'Mumbai', 'Rahul Mehta']
5 ['Delhi', 'INC-7', 'Identity', 'Login', 'Sara Khan']


## Try next

Add passages and relations of your own and see which walks change. Never paste a secret into a notebook cell.